# 01. EDA & Thống kê Bộ dữ liệu PhoNER_COVID19

Notebook của **Đạt**: tải tự động bộ dữ liệu `PhoNER_COVID19`, thống kê số câu, số token, phân phối độ dài câu, và tần suất 10 loại thực thể trên `train` / `dev` / `test` ở cả hai mức `word` và `syllable`, đồng thời xuất biểu đồ vào `outputs/figures/`.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.download import download_phoner_covid19, load_phoner_split
from src.data.labels import ENTITY_TYPES
from src.evaluation.error_analysis import extract_entities

DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

download_phoner_covid19(DATA_DIR)

In [ ]:
stats_rows = []
entity_freq = {}
word_lengths = {}

for level in ["word", "syllable"]:
    entity_freq[level] = {}
    for split in ["train", "dev", "test"]:
        records = load_phoner_split(DATA_DIR, level=level, split=split)
        lengths = [len(r["words"]) for r in records]
        if level == "word":
            word_lengths[split] = lengths
        counts = Counter()
        for r in records:
            for ent_type, _, _ in extract_entities(r["tags"]):
                counts[ent_type] += 1
        entity_freq[level][split] = counts
        stats_rows.append({
            "level": level,
            "split": split,
            "sentences": len(records),
            "tokens": sum(lengths),
            "entities": sum(counts.values()),
            "mean_len": float(np.mean(lengths)),
            "p95_len": float(np.percentile(lengths, 95)),
            "max_len": int(np.max(lengths)),
        })

for row in stats_rows:
    print(
        f"{row['level']:8s} | {row['split']:5s} | sents={row['sentences']:5d} | "
        f"tokens={row['tokens']:6d} | entities={row['entities']:5d} | "
        f"mean={row['mean_len']:.1f} | p95={row['p95_len']:.0f} | max={row['max_len']}"
    )

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for split in ["train", "dev", "test"]:
    sns.kdeplot(word_lengths[split], ax=axes[0], label=split, fill=True, alpha=0.25)
axes[0].set_title("Sentence Length Distribution (Word-level)")
axes[0].set_xlabel("Number of Words")
axes[0].legend()

x = np.arange(len(ENTITY_TYPES))
width = 0.27
for idx, split in enumerate(["train", "dev", "test"]):
    vals = [entity_freq["word"][split][etype] for etype in ENTITY_TYPES]
    axes[1].bar(x + (idx - 1) * width, vals, width=width, label=split)

axes[1].set_title("Entity Frequency across Splits (Word-level)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(ENTITY_TYPES, rotation=35, ha="right")
axes[1].set_ylabel("Entity Count")
axes[1].legend()

plt.tight_layout()
out_fig = FIG_DIR / "eda_data_distribution.png"
plt.savefig(out_fig, dpi=200)
print(f"Saved figure to {out_fig}")
plt.show()